<a href="https://colab.research.google.com/github/RioDeMilo/RETAIL-PRICES-COLOMBIA/blob/main/sipsa2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import csv
import json
import zeep
import xmltodict
import os

def extract_data(wsdl: str, service_method: str, **kwargs) -> list:
    """
    Dynamically calls the SOAP service method.
    """
    try:
        client = zeep.Client(wsdl=wsdl)
        # getattr finds the method by name (e.g., 'promediosSipsaCiudad')
        method = getattr(client.service, service_method, None)

        if not method:
            raise ValueError(f"Method '{service_method}' not found in WSDL.")

        # Pass any extra arguments (like arg0) directly to the service
        response = method(**kwargs)

        # Ensure we always return a list
        if response is None:
            return []
        return list(response) if isinstance(response, list) else [response]

    except zeep.exceptions.Fault as e:
        print(f"SOAP Fault: {e}")
        return []

def transform_data(raw_data: list, fields: list) -> list:
    """
    Filters data to include only requested fields and converts values to strings.
    """
    transformed = []
    for record in raw_data:
        row = {field: str(record[field]) for field in fields if field in record}
        transformed.append(row)
    return transformed
    # Instead of this I want to try other type since dataframe in pandas
    # is a list I could use that to create the dataframe
    # and work my way up to specific function for every type of database
    # df = pd.dataframe(raw_data)
    # Here I could export the data to csv
    # pandas.DataFrame.to_csv(path_file,readotheroptions)

def save_csv(path_file: str, fields: list, data: list):
    # DictWriter automatically handles header mapping and commas in data
    with open(path_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fields)
        writer.writeheader()
        writer.writerows(data)


def controller(wsdl: str, service_method: str, fields: list, path_file: str, **kwargs):
    print(f">>> Connecting to: {service_method}")

    # 1. Extract
    raw_data = extract_data(wsdl, service_method, **kwargs)

    if not raw_data:
        print(">>> No data returned from service.")
        return

    print(f">>> Retrieved {len(raw_data)} records.")

    # 2. Transform
    clean_data = transform_data(raw_data, fields)

    # 3. Load (Save)
    ext = os.path.splitext(path_file)[1].lower()

    print(f">>> Saving to {ext}...")
    if ext == '.json':
        save_json(path_file, clean_data)
    elif ext == '.csv':
        save_csv(path_file, fields, clean_data)
    elif ext == '.xml':
        save_xml(path_file, clean_data)
    else:
        print(f"Error: Unsupported file extension '{ext}'")

    print(f">>> Done! File saved at: {path_file}")


ModuleNotFoundError: No module named 'zeep'

In [ ]:
#Import the file from google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a link named 'sipsa_data.csv' in the local '/content/' directory
!ln -s "/content/drive/MyDrive/Colab Notebooks/promediosSipsaCiudad.csv" /content/sipsa_data.csv

# Now access the file at: /content/sipsa_data.csv
import numpy as np
import pandas as pd
df = pd.read_csv('/content/sipsa_data.csv')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ln: failed to create symbolic link '/content/sipsa_data.csv': File exists


In [ ]:
## when writing code this can be omitted
## First I will detect the encoding that gived us the previous libray using the chardet library
import chardet as char
# Step 2: Read CSV File in Binary Mode
with open('/content/sipsa_data.csv', 'rb') as f:
    data = f.read()

# Step 3: Detect Encoding using chardet Library
encoding_result = char.detect(data)

# Step 4: Retrieve Encoding Information
encoding = encoding_result['encoding']

# Step 5: Print Detected Encoding Information
print("Detected Encoding:", encoding)




Detected Encoding: utf-8


In [ ]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338017 entries, 0 to 338016
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype                    
---  ------          --------------   -----                    
 0   ciudad          338017 non-null  object                   
 1   codProducto     338017 non-null  int64                    
 2   enviado         338017 non-null  int64                    
 3   fechaCaptura    338017 non-null  datetime64[ns, UTC-05:00]
 4   fechaCreacion   338017 non-null  object                   
 5   precioPromedio  338017 non-null  int64                    
 6   producto        338017 non-null  object                   
 7   regId           338017 non-null  int64                    
dtypes: datetime64[ns, UTC-05:00](1), int64(4), object(3)
memory usage: 20.6+ MB


,ciudad,codProducto,enviado,fechaCaptura,fechaCreacion,precioPromedio,producto,regId
0,Bucaramanga,4,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,1875,Cebolla junca,648981
1,Bucaramanga,5,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,1573,Chócolo mazorca,648982
2,Bucaramanga,6,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,6000,Habichuela,648983
3,Bucaramanga,7,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,3600,Pepino cohombro,648984
4,Bucaramanga,8,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,2825,Pimentón,648985


In [ ]:
"From the ETL code before converting to csv I need to drop the enviado, Fecha creacion."
"for the pandas python code I need to Normalize the ciudad and producto, convert to only date the fechaCaptura "
"and convert every column to its specific data type?"
df['fechaCaptura'].head()

,fechaCaptura
0,2026-01-20 00:00:00-05:00
1,2026-01-20 00:00:00-05:00
2,2026-01-20 00:00:00-05:00
3,2026-01-20 00:00:00-05:00
4,2026-01-20 00:00:00-05:00


In [ ]:
df["enviado"].isna().value_counts() #with this I can check if there are any null values
df["ciudad"].value_counts() #with this other I cansee the unique values

,count
ciudad,
"Bogotá, d.c.",38483
Medellín,38345
Pereira,33739
Bucaramanga,21073
Armenia,20836
Cali,20810
Barranquilla,16897
San josé de cúcuta,13776
Villavicencio,13619


In [ ]:

df["fechaCaptura"] = pd.to_datetime(df["fechaCaptura"],)

In [ ]:
df.head()

,ciudad,codProducto,enviado,fechaCaptura,fechaCreacion,precioPromedio,producto,regId
0,BUCARAMANGA,4,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,1875,Cebolla junca,648981
1,BUCARAMANGA,5,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,1573,Chócolo mazorca,648982
2,BUCARAMANGA,6,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,6000,Habichuela,648983
3,BUCARAMANGA,7,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,3600,Pepino cohombro,648984
4,BUCARAMANGA,8,0,2026-01-20 00:00:00-05:00,2026-01-20 14:00:01-05:00,2825,Pimentón,648985


In [ ]:
df['fechaCaptura'].head()

,fechaCaptura
0,2026-01-20 00:00:00-05:00
1,2026-01-20 00:00:00-05:00
2,2026-01-20 00:00:00-05:00
3,2026-01-20 00:00:00-05:00
4,2026-01-20 00:00:00-05:00


In [ ]:
# creating a dictionary to change the values of the cities

ciudades_dictonario = {'BOGOTÁ, D.C.': 'Bogotá', 'SAN JOSÉ DE CÚCUTA': 'Cúcuta','CARTAGENA DE INDIAS' : 'Cartagena'}
df['ciudad'] = df['ciudad'].replace(ciudades_dictonario)

In [ ]:
# create a dictionary to better correct the names of the cities so I can avoid problems in the future


df['ciudad'] = df['ciudad'].str.capitalize()
df['ciudad'].value_counts()

,count
ciudad,
Bogotá,38483
Medellín,38345
Pereira,33739
Cúcuta,21757
Bucaramanga,21073
Armenia,20836
Cali,20810
Barranquilla,16897
Villavicencio,13619
